# 01 EDA and Preprocessing

Dataset: official UCI Heart Disease `processed.cleveland.data`.
Target binarization: `0 -> normal`, `1..4 -> heart disease`.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd().parents[0] if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(ROOT))

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(ROOT / 'src'))
from preprocessing import CONTINUOUS_FEATURES, FEATURE_COLUMNS, TARGET_COLUMN, build_preprocessor, load_heart_data, validate_clinical_ranges

df = load_heart_data()
df.head()

In [ ]:
df.info()
df.describe()

In [ ]:
target_distribution = df[TARGET_COLUMN].value_counts(normalize=True).rename('proportion')
target_distribution

The class balance affects metric choice. Because missing a true heart-disease case is clinically more costly than a false alarm, the project reports balanced accuracy and recall, not only plain accuracy.

In [ ]:
missing_by_column = df.isna().sum()
missing_total = int(df.isna().sum().sum())
missing_by_column, missing_total

In [ ]:
fig, axes = plt.subplots(1, len(CONTINUOUS_FEATURES), figsize=(16, 3))
for ax, column in zip(axes, CONTINUOUS_FEATURES):
    sns.boxplot(y=df[column], ax=ax)
    ax.set_title(column)
fig.tight_layout()

In [ ]:
iqr_rows = []
for column in CONTINUOUS_FEATURES:
    q1 = df[column].quantile(0.25)
    q3 = df[column].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    count = int(((df[column] < lower) | (df[column] > upper)).sum())
    iqr_rows.append({'feature': column, 'lower': lower, 'upper': upper, 'outlier_count': count})
pd.DataFrame(iqr_rows)

In [ ]:
duplicate_count = int(df.duplicated().sum())
validated = validate_clinical_ranges(df[FEATURE_COLUMNS])
preprocessor = build_preprocessor()
transformed = preprocessor.fit_transform(validated)
duplicate_count, transformed.shape

## EDA -> preprocessing decisions

- Missing values are handled inside a reusable sklearn `Pipeline`: median imputation for continuous features and most-frequent imputation for categorical coded features.
- Continuous features are standardized after imputation to support linear and kernel models without leakage; the transformer is fit only on training folds in `src/train.py`.
- Categorical coded features are one-hot encoded with `handle_unknown='ignore'` so new inference rows can be processed.
- Clinical range validation is kept as a reusable transformer so invalid values such as `chol > 600` are rejected before inference.
- Outliers are inspected but not blindly removed because extreme clinical values may be meaningful risk signals.